# EVChargeIQ — Data Audit

## Public EV Charging Transactions & Infrastructure

### Objective

This notebook performs the initial data audit for the EVChargeIQ project.

The audit examines:

- Dataset structure and scale
- Station-level infrastructure data
- Charging transaction data
- Data types and schema consistency
- Missing values
- Duplicate records
- Categorical fields
- Timestamp consistency
- Duration consistency
- Financial consistency
- Zero-value transactions
- Derived charging-power behavior

# Dataset Scope

The dataset contains public EV charging transaction and infrastructure data for Beijing, China, covering:

- January 2025
- July 2025

The two months are analyzed as observed periods rather than as a complete annual time series.

# Data Sources

- `stations_public.parquet`
- `orders_2025-01_public.parquet`
- `orders_2025-07_public.parquet`

---

**Project:** EVChargeIQ
**Stage:** Data Audit
**Tools:** Python, Pandas, NumPy, PyArrow

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 31952289.zip to 31952289.zip


In [ ]:
import os

print(os.listdir("/content"))

['.config', '31952289.zip', 'sample_data']


In [ ]:
import zipfile
import os

zip_path = "/content/31952289.zip"
extract_path = "/content/EVChargeIQ_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
print(os.listdir("/content/EVChargeIQ_data"))

['orders_2025-07_public.parquet', 'stations_public.parquet', 'code.ipynb', 'orders_2025-01_public.parquet']


In [ ]:
import os
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
BASE_PATH = "/content/EVChargeIQ_data"

stations_path = f"{BASE_PATH}/stations_public.parquet"
jan_path = f"{BASE_PATH}/orders_2025-01_public.parquet"
jul_path = f"{BASE_PATH}/orders_2025-07_public.parquet"

print("Dataset paths configured.")
print("\nStation file:", stations_path)
print("January file:", jan_path)
print("July file:", jul_path)

Dataset paths configured.

Station file: /content/EVChargeIQ_data/stations_public.parquet
January file: /content/EVChargeIQ_data/orders_2025-01_public.parquet
July file: /content/EVChargeIQ_data/orders_2025-07_public.parquet


In [ ]:
required_files = [
    stations_path,
    jan_path,
    jul_path
]

for file_path in required_files:
    print(
        os.path.basename(file_path),
        "→",
        "FOUND" if os.path.exists(file_path) else "NOT FOUND"
    )

stations_public.parquet → FOUND
orders_2025-01_public.parquet → FOUND
orders_2025-07_public.parquet → FOUND


In [11]:
# Read Parquet metadata without loading the full datasets

stations_file = pq.ParquetFile(stations_path)
jan_file = pq.ParquetFile(jan_path)
jul_file = pq.ParquetFile(jul_path)

print("STATION DATASET")
print("Rows:", stations_file.metadata.num_rows)
print("Columns:", stations_file.metadata.num_columns)

print("\nJANUARY 2025 TRANSACTIONS")
print("Rows:", jan_file.metadata.num_rows)
print("Columns:", jan_file.metadata.num_columns)

print("\nJULY 2025 TRANSACTIONS")
print("Rows:", jul_file.metadata.num_rows)
print("Columns:", jul_file.metadata.num_columns)

print("\nTOTAL TRANSACTIONS:",
      jan_file.metadata.num_rows + jul_file.metadata.num_rows)

STATION DATASET
Rows: 8553
Columns: 10

JANUARY 2025 TRANSACTIONS
Rows: 2137234
Columns: 12

JULY 2025 TRANSACTIONS
Rows: 6407461
Columns: 12

TOTAL TRANSACTIONS: 8544695


In [12]:
print("STATION SCHEMA")
print(stations_file.schema_arrow)

print("\nJANUARY TRANSACTION SCHEMA")
print(jan_file.schema_arrow)

print("\nJULY TRANSACTION SCHEMA")
print(jul_file.schema_arrow)

STATION SCHEMA
geocoding: string
station_id: string
construction_site: string
station_total_power_kw: double
piles_num: double
dc_piles_num: double
ac_piles_num: double
charging_gun_num: double
dc_charging_gun_num: double
ac_charging_gun_num: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1372

JANUARY TRANSACTION SCHEMA
transaction_date: date32[day]
equipment_classification: string
rated_gun_power_kw: double
station_id: string
pile_id: string
duration_min: double
charge_start_time: timestamp[ns]
charge_end_time: timestamp[ns]
total_elec_kwh: double
total_elec_fee: double
total_service_fee: double
total_fee: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1642

JULY TRANSACTION SCHEMA
transaction_date: date32[day]
equipment_classification: string
rated_gun_power_kw: double
station_id: string
pile_id: string
duration_min: double
charge_start_time: timestamp[ns]
charge_end

In [13]:
# Load station-level data

stations = pd.read_parquet(stations_path)

print("Station dataset loaded successfully.")
print("Shape:", stations.shape)

Station dataset loaded successfully.
Shape: (8553, 10)


In [14]:
stations.head()

,geocoding,station_id,construction_site,station_total_power_kw,piles_num,dc_piles_num,ac_piles_num,charging_gun_num,dc_charging_gun_num,ac_charging_gun_num
0,9891_4446,ST_000001,居住区,21.0,3.0,0.0,3.0,3.0,0.0,3.0
1,9911_4435,ST_000002,居住区,42.0,6.0,0.0,6.0,6.0,0.0,6.0
2,9911_4435,ST_000003,居民区,21.0,3.0,0.0,3.0,3.0,0.0,3.0
3,9904_4440,ST_000004,居住区,84.0,6.0,0.0,6.0,12.0,0.0,12.0
4,9906_4417,ST_000005,居住区,87.0,6.0,0.0,6.0,6.0,0.0,6.0


## 1. Station Dataset Structure

The station dataset contains infrastructure-level attributes for public EV charging stations.

The `station_id` field serves as the common identifier used to connect station-level information with transaction-level charging records.

In [15]:
print("Station columns:")
for i, column in enumerate(stations.columns, start=1):
    print(f"{i}. {column}")

Station columns:
1. geocoding
2. station_id
3. construction_site
4. station_total_power_kw
5. piles_num
6. dc_piles_num
7. ac_piles_num
8. charging_gun_num
9. dc_charging_gun_num
10. ac_charging_gun_num


In [16]:
stations.dtypes

,0
geocoding,object
station_id,object
construction_site,object
station_total_power_kw,float64
piles_num,float64
dc_piles_num,float64
ac_piles_num,float64
charging_gun_num,float64
dc_charging_gun_num,float64
ac_charging_gun_num,float64


### Missing-Value Check

Missing values are checked across all station-level attributes before any transformation or analysis.

In [17]:
station_missing = stations.isnull().sum()

print("Missing values by column:")
print(station_missing)

print("\nTotal missing values:",
      station_missing.sum())

Missing values by column:
geocoding                 0
station_id                0
construction_site         0
station_total_power_kw    0
piles_num                 0
dc_piles_num              0
ac_piles_num              0
charging_gun_num          0
dc_charging_gun_num       0
ac_charging_gun_num       0
dtype: int64

Total missing values: 0


In [18]:
total_station_ids = stations["station_id"].nunique()
duplicate_station_ids = stations["station_id"].duplicated().sum()

print("Total station IDs:", total_station_ids)
print("Duplicate station IDs:", duplicate_station_ids)

Total station IDs: 8553
Duplicate station IDs: 0


## 2. Construction-Site Categories

The `construction_site` field describes the primary scenario associated with each charging station.

The raw dataset contains categorical values in Chinese. These values are preserved in the raw data and will be standardized into English labels later during the data-preparation stage.

In [19]:
construction_site_counts = (
    stations["construction_site"]
    .value_counts(dropna=False)
)

print("Unique construction-site categories:",
      stations["construction_site"].nunique())

construction_site_counts

Unique construction-site categories: 17


,count
construction_site,
其他,2577
公共机构,1791
居住区,1208
企事业单位,1093
大型建筑配建停车场,533
写字楼,501
居民区,339
工业园区,159
大型文体设施,121


### Station Infrastructure Statistics

The following summary describes station power capacity, charging-pile counts, and charging-gun counts.

Extreme values are identified for investigation but are not automatically removed during the audit.

In [20]:
station_numeric_cols = [
    "station_total_power_kw",
    "piles_num",
    "dc_piles_num",
    "ac_piles_num",
    "charging_gun_num",
    "dc_charging_gun_num",
    "ac_charging_gun_num"
]

stations[station_numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
station_total_power_kw,8553.0,497.920262,1116.400282,3.5,44.0,140.0,480.0,18120.0
piles_num,8553.0,9.143692,15.694599,1.0,3.0,6.0,10.0,876.0
dc_piles_num,8553.0,3.460540,6.122626,0.0,0.0,1.0,4.0,96.0
ac_piles_num,8553.0,5.703964,14.947251,0.0,0.0,3.0,6.0,876.0
charging_gun_num,8553.0,11.501929,17.283446,1.0,4.0,8.0,12.0,876.0
dc_charging_gun_num,8553.0,5.437975,9.916721,0.0,0.0,2.0,7.0,220.0
ac_charging_gun_num,8553.0,6.094821,15.274592,0.0,0.0,3.0,7.0,876.0


In [21]:
print("Pile-count consistency:")
print(
    "Rows where DC piles + AC piles != total piles:",
    (
        stations["dc_piles_num"] +
        stations["ac_piles_num"]
        != stations["piles_num"]
    ).sum()
)

print("\nCharging-gun consistency:")
print(
    "Rows where DC guns + AC guns != total guns:",
    (
        stations["dc_charging_gun_num"] +
        stations["ac_charging_gun_num"]
        != stations["charging_gun_num"]
    ).sum()
)

Pile-count consistency:
Rows where DC piles + AC piles != total piles: 41

Charging-gun consistency:
Rows where DC guns + AC guns != total guns: 42


In [22]:
print(
    "Stations with >= 500 piles:",
    (stations["piles_num"] >= 500).sum()
)

print(
    "Stations with >= 1,000 kW:",
    (stations["station_total_power_kw"] >= 1000).sum()
)

Stations with >= 500 piles: 1
Stations with >= 1,000 kW: 1032


In [23]:
stations[
    stations["piles_num"] >= 500
][
    [
        "station_id",
        "construction_site",
        "station_total_power_kw",
        "piles_num",
        "dc_piles_num",
        "ac_piles_num",
        "charging_gun_num",
        "dc_charging_gun_num",
        "ac_charging_gun_num"
    ]
]

,station_id,construction_site,station_total_power_kw,piles_num,dc_piles_num,ac_piles_num,charging_gun_num,dc_charging_gun_num,ac_charging_gun_num
3777,ST_003778,其他,6132.0,876.0,0.0,876.0,876.0,0.0,876.0


## 3. Station Audit Summary

### Key Findings

- The station dataset contains **8,553 stations** across **10 fields**.
- All station fields contain **zero missing values**.
- `station_id` is unique across all 8,553 station records.
- The `construction_site` field contains **17 categorical values**.
- Station infrastructure includes total power, charging piles, and charging guns with separate AC/DC counts.
- One station has at least 500 charging piles and is treated as an extreme observation rather than automatically removed.
- High-capacity stations are retained because extreme infrastructure values may represent legitimate large-scale facilities.

### Data-Preparation Decision

No station records are removed during the initial audit.

The original Chinese categorical values are preserved and will be mapped to standardized English categories during feature engineering.

## 4. Transaction Dataset Audit

The transaction data contains more than 8.5 million charging records across January and July 2025.

Because of the dataset size, the initial audit uses Parquet metadata and representative samples rather than repeatedly loading the complete transaction tables into memory.

In [25]:
transaction_summary = pd.DataFrame({
    "period": ["January 2025", "July 2025"],
    "transactions": [
        jan_file.metadata.num_rows,
        jul_file.metadata.num_rows
    ]
})

transaction_summary.loc[len(transaction_summary)] = [
    "Total",
    transaction_summary["transactions"].sum()
]

transaction_summary

,period,transactions
0,January 2025,2137234
1,July 2025,6407461
2,Total,8544695


## 5. Transaction Sampling

The transaction tables contain more than 8.5 million records. A representative sample is used for initial quality checks to avoid unnecessary memory consumption during exploratory auditing.

The full transaction files remain stored in Parquet format and will be processed efficiently during later analysis.

In [26]:
# Read a 10,000-row sample from each transaction file

SAMPLE_SIZE = 10_000

jan_sample = pd.read_parquet(
    jan_path
).head(SAMPLE_SIZE)

jul_sample = pd.read_parquet(
    jul_path
).head(SAMPLE_SIZE)

print("January sample shape:", jan_sample.shape)
print("July sample shape:", jul_sample.shape)

January sample shape: (10000, 12)
July sample shape: (10000, 12)


In [27]:
print("January transaction columns:")
print(jan_sample.columns.tolist())

print("\nJuly transaction columns:")
print(jul_sample.columns.tolist())

print(
    "\nSchemas identical:",
    list(jan_sample.columns) == list(jul_sample.columns)
)

January transaction columns:
['transaction_date', 'equipment_classification', 'rated_gun_power_kw', 'station_id', 'pile_id', 'duration_min', 'charge_start_time', 'charge_end_time', 'total_elec_kwh', 'total_elec_fee', 'total_service_fee', 'total_fee']

July transaction columns:
['transaction_date', 'equipment_classification', 'rated_gun_power_kw', 'station_id', 'pile_id', 'duration_min', 'charge_start_time', 'charge_end_time', 'total_elec_kwh', 'total_elec_fee', 'total_service_fee', 'total_fee']

Schemas identical: True


In [28]:
print("January dtypes:")
print(jan_sample.dtypes)

print("\nJuly dtypes:")
print(jul_sample.dtypes)

January dtypes:
transaction_date                    object
equipment_classification            object
rated_gun_power_kw                 float64
station_id                          object
pile_id                             object
duration_min                       float64
charge_start_time           datetime64[ns]
charge_end_time             datetime64[ns]
total_elec_kwh                     float64
total_elec_fee                     float64
total_service_fee                  float64
total_fee                          float64
dtype: object

July dtypes:
transaction_date                    object
equipment_classification            object
rated_gun_power_kw                 float64
station_id                          object
pile_id                             object
duration_min                       float64
charge_start_time           datetime64[ns]
charge_end_time             datetime64[ns]
total_elec_kwh                     float64
total_elec_fee                     float64
total_serv

### Equipment Classification

The transaction dataset contains two charging-equipment categories.

The original Chinese values are preserved during the audit and will be translated into standardized English labels during the data-preparation stage.

In [30]:
print("January equipment classifications:")
print(
    jan_sample["equipment_classification"]
    .value_counts(dropna=False)
)

print("\nJuly equipment classifications:")
print(
    jul_sample["equipment_classification"]
    .value_counts(dropna=False)
)

January equipment classifications:
equipment_classification
直流设备    7076
交流设备    2924
Name: count, dtype: int64

July equipment classifications:
equipment_classification
直流设备    9084
交流设备     916
Name: count, dtype: int64


### Missing-Value Audit

Missing values are checked across the transaction sample before any transformations are applied.

In [31]:
jan_missing = jan_sample.isnull().sum()
jul_missing = jul_sample.isnull().sum()

print("January missing values:")
print(jan_missing)

print("\nJuly missing values:")
print(jul_missing)

print("\nTotal January missing values:", jan_missing.sum())
print("Total July missing values:", jul_missing.sum())

January missing values:
transaction_date            0
equipment_classification    0
rated_gun_power_kw          0
station_id                  0
pile_id                     0
duration_min                0
charge_start_time           0
charge_end_time             0
total_elec_kwh              0
total_elec_fee              0
total_service_fee           0
total_fee                   0
dtype: int64

July missing values:
transaction_date            0
equipment_classification    0
rated_gun_power_kw          0
station_id                  0
pile_id                     0
duration_min                0
charge_start_time           0
charge_end_time             0
total_elec_kwh              0
total_elec_fee              0
total_service_fee           0
total_fee                   0
dtype: int64

Total January missing values: 0
Total July missing values: 0


In [32]:
print(
    "January duplicate rows:",
    jan_sample.duplicated().sum()
)

print(
    "July duplicate rows:",
    jul_sample.duplicated().sum()
)

January duplicate rows: 0
July duplicate rows: 0


### Timestamp Consistency

The transaction records contain charging start and end timestamps.

The following checks verify that:

1. Charging ends after it starts.
2. Recorded duration matches the elapsed time between the timestamps.

In [33]:
# Calculate elapsed duration from timestamps

jan_sample["timestamp_duration_min"] = (
    jan_sample["charge_end_time"] -
    jan_sample["charge_start_time"]
).dt.total_seconds() / 60

jul_sample["timestamp_duration_min"] = (
    jul_sample["charge_end_time"] -
    jul_sample["charge_start_time"]
).dt.total_seconds() / 60

print(
    "January invalid timestamp order:",
    (jan_sample["timestamp_duration_min"] <= 0).sum()
)

print(
    "July invalid timestamp order:",
    (jul_sample["timestamp_duration_min"] <= 0).sum()
)

January invalid timestamp order: 0
July invalid timestamp order: 0


In [34]:
jan_duration_diff = (
    jan_sample["duration_min"] -
    jan_sample["timestamp_duration_min"]
)

jul_duration_diff = (
    jul_sample["duration_min"] -
    jul_sample["timestamp_duration_min"]
)

print("January duration difference:")
print(jan_duration_diff.describe())

print("\nJuly duration difference:")
print(jul_duration_diff.describe())

January duration difference:
count    10000.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
dtype: float64

July duration difference:
count    10000.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
dtype: float64


### Numerical Validity

Energy, duration, rated power, and fee fields are checked for negative values.

Negative values would require investigation because they are not physically or financially meaningful for these transaction measures.

In [35]:
transaction_numeric_cols = [
    "rated_gun_power_kw",
    "duration_min",
    "total_elec_kwh",
    "total_elec_fee",
    "total_service_fee",
    "total_fee"
]

print("JANUARY NEGATIVE VALUES")
print(
    (jan_sample[transaction_numeric_cols] < 0).sum()
)

print("\nJULY NEGATIVE VALUES")
print(
    (jul_sample[transaction_numeric_cols] < 0).sum()
)

JANUARY NEGATIVE VALUES
rated_gun_power_kw    0
duration_min          0
total_elec_kwh        0
total_elec_fee        0
total_service_fee     0
total_fee             0
dtype: int64

JULY NEGATIVE VALUES
rated_gun_power_kw    0
duration_min          0
total_elec_kwh        0
total_elec_fee        0
total_service_fee     0
total_fee             0
dtype: int64


### Zero-Value Analysis

Zero values are identified separately from negative values.

A zero value is not automatically treated as an error because some charging sessions may legitimately have zero energy or zero fee components.

In [36]:
print("JANUARY ZERO VALUES")
print(
    (jan_sample[transaction_numeric_cols] == 0).sum()
)

print("\nJULY ZERO VALUES")
print(
    (jul_sample[transaction_numeric_cols] == 0).sum()
)

JANUARY ZERO VALUES
rated_gun_power_kw      0
duration_min            0
total_elec_kwh          1
total_elec_fee          8
total_service_fee     242
total_fee               7
dtype: int64

JULY ZERO VALUES
rated_gun_power_kw    0
duration_min          0
total_elec_kwh        2
total_elec_fee        2
total_service_fee     2
total_fee             2
dtype: int64


In [37]:
zero_energy_columns = [
    "transaction_date",
    "equipment_classification",
    "rated_gun_power_kw",
    "duration_min",
    "charge_start_time",
    "charge_end_time",
    "total_elec_kwh",
    "total_elec_fee",
    "total_service_fee",
    "total_fee"
]

print("January zero-energy transactions:")
display(
    jan_sample[
        jan_sample["total_elec_kwh"] == 0
    ][zero_energy_columns]
)

print("\nJuly zero-energy transactions:")
display(
    jul_sample[
        jul_sample["total_elec_kwh"] == 0
    ][zero_energy_columns]
)

January zero-energy transactions:


,transaction_date,equipment_classification,rated_gun_power_kw,duration_min,charge_start_time,charge_end_time,total_elec_kwh,total_elec_fee,total_service_fee,total_fee
6203,2025-01-12,直流设备,90.0,0.116667,2025-01-12 03:17:32,2025-01-12 03:17:39,0.0,0.0,0.0,0.0



July zero-energy transactions:


,transaction_date,equipment_classification,rated_gun_power_kw,duration_min,charge_start_time,charge_end_time,total_elec_kwh,total_elec_fee,total_service_fee,total_fee
1953,2025-07-01,直流设备,240.0,71.000000,2025-07-01 01:10:18,2025-07-01 02:21:18,0.0,0.0,0.0,0.0
4948,2025-07-02,直流设备,240.0,86.233333,2025-07-02 21:53:02,2025-07-02 23:19:16,0.0,0.0,0.0,0.0


### Financial Consistency

The transaction-level financial fields are expected to satisfy:

`total_fee = total_elec_fee + total_service_fee`

The difference is evaluated with a small tolerance to account for floating-point representation.

In [38]:
jan_sample["fee_difference"] = (
    jan_sample["total_fee"]
    - jan_sample["total_elec_fee"]
    - jan_sample["total_service_fee"]
)

jul_sample["fee_difference"] = (
    jul_sample["total_fee"]
    - jul_sample["total_elec_fee"]
    - jul_sample["total_service_fee"]
)

print("January fee difference:")
print(jan_sample["fee_difference"].describe())

print("\nJuly fee difference:")
print(jul_sample["fee_difference"].describe())

January fee difference:
count    1.000000e+04
mean     4.992205e-17
std      4.198210e-15
min     -2.131628e-14
25%     -1.776357e-15
50%      0.000000e+00
75%      1.776357e-15
max      2.842171e-14
Name: fee_difference, dtype: float64

July fee difference:
count    1.000000e+04
mean    -1.705944e-17
std      2.345779e-15
min     -2.131628e-14
25%     -8.881784e-16
50%      0.000000e+00
75%      8.881784e-16
max      2.842171e-14
Name: fee_difference, dtype: float64


In [39]:
FEE_TOLERANCE = 0.01

jan_fee_mismatch = (
    jan_sample["fee_difference"].abs() > FEE_TOLERANCE
).sum()

jul_fee_mismatch = (
    jul_sample["fee_difference"].abs() > FEE_TOLERANCE
).sum()

print(
    "January fee mismatches:",
    jan_fee_mismatch
)

print(
    "July fee mismatches:",
    jul_fee_mismatch
)

January fee mismatches: 0
July fee mismatches: 0


### Derived Average Charging Power

A session-level estimate of average charging power is calculated as:

`average charging power = total energy / charging duration`

This is treated as a derived analytical metric rather than a data-validation rule.

The comparison with `rated_gun_power_kw` is not used to remove records because the observed relationship does not provide sufficient evidence that exceeding the rated value represents an invalid transaction.

In [40]:
jan_sample["avg_charging_power_kw"] = (
    jan_sample["total_elec_kwh"] /
    (jan_sample["duration_min"] / 60)
)

jul_sample["avg_charging_power_kw"] = (
    jul_sample["total_elec_kwh"] /
    (jul_sample["duration_min"] / 60)
)

print("January average charging power:")
print(jan_sample["avg_charging_power_kw"].describe())

print("\nJuly average charging power:")
print(jul_sample["avg_charging_power_kw"].describe())

January average charging power:
count    10000.000000
mean        60.896868
std         56.650815
min          0.000000
25%          6.991477
50%         49.826584
75%         90.348880
max        439.240223
Name: avg_charging_power_kw, dtype: float64

July average charging power:
count    10000.000000
mean        40.977822
std         27.537526
min          0.000000
25%         25.408043
50%         36.000000
75%         51.588233
max        338.636066
Name: avg_charging_power_kw, dtype: float64


In [41]:
jan_exceed_count = (
    jan_sample["avg_charging_power_kw"] >
    jan_sample["rated_gun_power_kw"]
).sum()

jul_exceed_count = (
    jul_sample["avg_charging_power_kw"] >
    jul_sample["rated_gun_power_kw"]
).sum()

print(
    "January sessions exceeding rated power:",
    jan_exceed_count
)

print(
    "July sessions exceeding rated power:",
    jul_exceed_count
)

January sessions exceeding rated power: 2584
July sessions exceeding rated power: 100


### Decision

A substantial number of sampled January transactions have calculated average charging power above the recorded rated gun power.

Because the reason for this relationship cannot be established from the available fields alone, these records are **not removed or modified**.

`avg_charging_power_kw` is retained as a derived exploratory metric only.

The original `rated_gun_power_kw` and `total_elec_kwh` values remain unchanged.

## 6. Final Data Audit Summary

### Station Dataset

| Check | Result |
|---|---:|
| Stations | 8,553 |
| Columns | 10 |
| Unique station IDs | 8,553 |
| Duplicate station IDs | 0 |
| Missing values | 0 |
| Construction-site categories | 17 |
| Pile-count inconsistencies | 41 |
| Charging-gun inconsistencies | 42 |

### Transaction Dataset

| Check | January | July |
|---|---:|---:|
| Transactions | 2,137,234 | 6,407,461 |
| Columns | 12 | 12 |
| Sample size | 10,000 | 10,000 |
| Missing values in sample | 0 | 0 |
| Duplicate rows in sample | 0 | 0 |
| Invalid timestamp order | 0 | 0 |
| Duration/timestamp mismatch | 0 | 0 |
| Negative numerical values | 0 | 0 |
| Fee reconciliation mismatches | 0 | 0 |

### Important Findings

1. The station and transaction datasets have compatible `station_id` fields for downstream analysis.
2. Both transaction months share the same schema.
3. The transaction data contains two equipment categories: DC and AC.
4. Chinese categorical values are retained in the raw layer.
5. Zero-energy transactions exist and are not automatically treated as invalid.
6. Financial fields reconcile within floating-point tolerance.
7. Station component-count inconsistencies exist in a small number of records and are retained rather than silently corrected.
8. Derived average charging power can exceed recorded rated power; this is treated as an analytical observation rather than a cleaning rule.

### Data Preparation Principle

**Raw source values will not be overwritten during cleaning.**

Any translations, derived metrics, flags, or analytical transformations will be created in separate processed datasets.